# Chapitre 7 · Découper le langage (solutions des exercices)

Ce notebook contient **uniquement les réponses aux quatre exercices** du notebook
du chapitre. Le code de la leçon, lui, vit dans le notebook du chapitre et dans le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte
« IA débranchée » vaut aussi pour les corrigés.

## Mise en place (reprise de la leçon)

Le minimum pour que les validations tournent en autonomie : le marqueur de fin de mot,
le mini-corpus de Sennrich, la fonction `preparer`, et l'historique de fusions appris
dans la leçon.

In [ ]:
from collections import Counter

FIN = "</w>"                                   # le marqueur de fin de mot

mini_corpus = "manger manger manger chanter chanter danser danser nager"

def preparer(corpus):
    frequences = Counter(corpus.split())       # mot -> nombre d'occurrences
    decoupes = {mot: list(mot) + [FIN] for mot in frequences}
    return frequences, decoupes

# L'historique appris dans la leçon (6 fusions sur le mini-corpus).
fusions_mini = [("e", "r"), ("er", FIN), ("a", "n"), ("g", "er</w>"), ("m", "an"), ("man", "ger</w>")]
print("Mise en place OK.")

### Exercice 1 · Compter les paires voisines — niveau ●

In [ ]:
frequences_ex, decoupes_ex = preparer(mini_corpus)

def compter_paires(frequences, decoupes):
    """Compte les paires de tokens voisins dans tout le corpus.

    Rend un Counter : (token_gauche, token_droit) -> nombre d'occurrences.
    """
    paires = Counter()
    for mot, freq in frequences.items():
        tokens = decoupes[mot]
        for i in range(len(tokens) - 1):
            paires[(tokens[i], tokens[i + 1])] += freq
    return paires

In [ ]:
# Validation : compter les paires.
paires = compter_paires(frequences_ex, decoupes_ex)
assert paires[("e", "r")] == 8, "attendu 8 : les quatre verbes finissent par 'er'. As-tu multiplié par freq ?"
assert paires[("a", "n")] == 7, "attendu 7 : manger (×3), chanter (×2), danser (×2)."
assert paires[("g", "e")] == 4, "attendu 4 : manger (×3) et nager (×1)."
assert len(paires) == 15, f"attendu 15 paires distinctes, obtenu {len(paires)}."
meilleure = max(paires, key=paires.get)
assert meilleure == ("e", "r"), f"la plus fréquente devrait être ('e', 'r'), obtenu {meilleure}."
print("Exercice 1 validé : 15 paires comptées, ('e', 'r') en tête avec 8 occurrences.")

### Exercice 2 · Appliquer une fusion — niveau ●●

In [ ]:
def fusionner(tokens, paire):
    """Remplace chaque occurrence de la paire (a, b) par le token collé a + b."""
    a, b = paire
    resultat = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
            resultat.append(a + b)             # les deux voisins deviennent un
            i += 2                             # on saute les deux d'un coup
        else:
            resultat.append(tokens[i])
            i += 1
    return resultat

In [ ]:
# Validation : appliquer une fusion.
assert fusionner(["m", "a", "n", "g", "e", "r", FIN], ("e", "r")) == ["m", "a", "n", "g", "er", FIN]
assert fusionner(list("baobab") + [FIN], ("b", "a")) == ["ba", "o", "ba", "b", FIN], \
    "la paire apparaît deux fois dans baobab : les deux occurrences doivent fusionner."
assert fusionner(["man", "ger", FIN], ("x", "y")) == ["man", "ger", FIN], "paire absente : rien ne change."
print("Exercice 2 validé : la fusion s'applique à toutes les occurrences, et à elles seules.")

### Exercice 3 · Encoder un mot nouveau — niveau ●●

In [ ]:
def encoder_mot(mot, fusions):
    """Découpe un mot (connu ou inconnu) en tokens du vocabulaire."""
    tokens = list(mot) + [FIN]
    for paire in fusions:                      # l'ordre appris, rejoué tel quel
        tokens = fusionner(tokens, paire)
    return tokens

In [ ]:
# Validation : encoder un mot nouveau.
assert encoder_mot("manger", fusions_mini) == ["manger</w>"]
assert encoder_mot("changer", fusions_mini) == ["c", "h", "an", "ger</w>"], \
    "changer n'est PAS dans le corpus, et pourtant : quatre morceaux connus. C'est toute la magie subword."
assert encoder_mot("cadet", fusions_mini) == ["c", "a", "d", "e", "t", FIN], \
    "aucune fusion ne s'applique : retombée propre sur les caractères."
print("Exercice 3 validé : encoder_mot('changer') ->", encoder_mot("changer", fusions_mini))

### Exercice 4 · La boucle d'entraînement BPE — niveau ●●●

In [ ]:
def entrainer_bpe(corpus, nb_fusions):
    """Apprend nb_fusions fusions sur le corpus. Rend (fusions, decoupes)."""
    frequences, decoupes = preparer(corpus)
    fusions = []                               # l'historique, dans l'ordre
    for _ in range(nb_fusions):
        paires = compter_paires(frequences, decoupes)
        if not paires:                         # plus rien à fusionner
            break
        meilleure = max(paires, key=paires.get)
        fusions.append(meilleure)
        for mot in decoupes:
            decoupes[mot] = fusionner(decoupes[mot], meilleure)
    return fusions, decoupes

In [ ]:
# Validation : la boucle d'entraînement complète.
fusions_ex, decoupes_ex2 = entrainer_bpe(mini_corpus, 6)
attendu = [("e", "r"), ("er", FIN), ("a", "n"), ("g", "er</w>"), ("m", "an"), ("man", "ger</w>")]
assert fusions_ex == attendu, f"historique inattendu : {fusions_ex}"
assert decoupes_ex2["manger"] == ["manger</w>"], "après 6 fusions, manger doit tenir en un seul token."
assert decoupes_ex2["chanter"] == ["c", "h", "an", "t", "er</w>"]
print("Exercice 4 validé : ton pipeline BPE complet tourne. Historique appris :")
for k, (a, b) in enumerate(fusions_ex, start=1):
    print(f"  {k}. {a} + {b} -> {a + b}")